# Global Moran's I and LISA

This analysis reads territorial KPIs exclusively from `dw.v_kpi_ageb`. The reference sample is the 2,348 city-core AGEBs of Mexico City (the main locality, `cve_loc = '0001'`, of each of the 16 alcaldías). Queen contiguity is row-standardized; its island is attached to its nearest neighbor. Row-standardized KNN-6 provides the sensitivity check. Every statistic uses 999 permutations with a fixed seed and α = 0.05. Spatial association does not establish causation.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from esda import Moran
from libpysal.weights import lag_spatial
from src.analysis.data import load_kpis
from src.analysis.spatial_weights import build_weights, local_clusters

MAPS = ROOT / 'outputs/maps'
FIGURES = ROOT / 'outputs/figures'
MAPS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
SEED = 42  # permutation inference is reproducible across runs

## Warehouse prerequisite

Start PostGIS and run the complete ETL before executing this notebook. Loading with `exclude_low_population=False` keeps the 2,348-AGEB weights universe (TEAM_PLAN §3.5). Rate indicators (`crime_rate_per_1k`, `pea_rate`) then drop the AGEBs with fewer than 100 residents, whose rates are unstable (TEAM_PLAN §3.1); density indicators keep every city-core AGEB.

In [2]:
kpis = load_kpis(city_core_only=True, exclude_low_population=False).set_index('cvegeo')
assert len(kpis) == 2_348, f'Expected 2,348 city-core AGEBs; got {len(kpis)}'
INDICATORS = {
    'business_density_km2': 'Business density',
    'crime_rate_per_1k': 'Crime rate per 1,000 residents',
    'pop_density_km2': 'Population density',
    'pea_rate': 'Economically active population rate',
}
RATE_INDICATORS = {'crime_rate_per_1k', 'pea_rate'}
display(kpis[list(INDICATORS) + ['low_population']].describe().T)

,count,mean,std,min,25%,50%,75%,max
business_density_km2,2348.0,775.803222,966.955874,0.0,308.615523,606.284741,978.160849,22207.142857
crime_rate_per_1k,2331.0,55.252641,749.075573,0.0,6.929991,10.132502,15.124555,28714.285714
pop_density_km2,2348.0,17074.992363,10098.564075,0.0,9948.259659,16280.504047,23033.651243,122599.277978
pea_rate,2321.0,0.640474,0.055549,0.0,0.612472,0.636965,0.667211,1.000000


## Global Moran's I

Each indicator uses its complete observations only (rate indicators also drop low-population AGEBs), and Queen and KNN-6 are built on the identical indicator-specific sample. The table reports observed I, analytical expectation, permutation z-score, pseudo p-value and sample size. `esda.Moran` takes no seed, so NumPy's seed is fixed before each test.

In [3]:
results = []
samples = {}
for column, label in INDICATORS.items():
    keep = kpis[column].notna()
    if column in RATE_INDICATORS:
        keep &= ~kpis['low_population']
    sample = kpis.loc[keep, [column, 'geom']].copy()
    if len(sample) < 7 or sample[column].nunique() < 2:
        print(f'Skipped {column}: insufficient variation or observations')
        continue
    samples[column] = sample
    values = sample[column].to_numpy(dtype=float)
    for kind, kind_label in [('queen', 'Queen'), ('knn', 'KNN-6')]:
        weights = build_weights(sample, kind=kind, k=6)
        np.random.seed(SEED)
        statistic = Moran(values, weights, permutations=999)
        results.append({
            'indicator': label, 'weights': kind_label, 'n': len(sample),
            'I': statistic.I, 'E[I]': statistic.EI,
            'z_sim': statistic.z_sim, 'p_sim': statistic.p_sim,
            'permutations': statistic.permutations,
        })
moran_results = pd.DataFrame(results)
display(moran_results.round(4))

,indicator,weights,n,I,E[I],z_sim,p_sim,permutations
0,Business density,Queen,2348,0.4618,-0.0004,38.2189,0.001,999
1,Business density,KNN-6,2348,0.4583,-0.0004,42.3570,0.001,999
2,"Crime rate per 1,000 residents",Queen,2297,0.2096,-0.0004,17.2936,0.001,999
3,"Crime rate per 1,000 residents",KNN-6,2297,0.1785,-0.0004,17.0411,0.001,999
4,Population density,Queen,2348,0.4342,-0.0004,36.6669,0.001,999
5,Population density,KNN-6,2348,0.4508,-0.0004,39.8869,0.001,999
6,Economically active population rate,Queen,2297,0.5063,-0.0004,42.0699,0.001,999
7,Economically active population rate,KNN-6,2297,0.4996,-0.0004,44.6113,0.001,999


In [4]:
for column, sample in samples.items():
    values = sample[column].to_numpy(dtype=float)
    z = (values - values.mean()) / values.std(ddof=0)
    weights = build_weights(sample, kind='queen')
    lag = lag_spatial(weights, z)
    slope = Moran(values, weights, permutations=999).I
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(z, lag, s=16, alpha=.65)
    xline = np.array([z.min(), z.max()])
    ax.plot(xline, slope * xline, color='crimson', label=f'I = {slope:.3f}')
    ax.axhline(0, color='0.6', linewidth=.8); ax.axvline(0, color='0.6', linewidth=.8)
    ax.set(xlabel='Standardized value', ylabel='Spatial lag', title=f"Moran scatterplot: {INDICATORS[column]}")
    ax.legend(); fig.tight_layout()
    fig.savefig(FIGURES / f'32_moran_{column}.png', dpi=180)
    plt.close(fig)

## Local Moran (LISA) clusters

The first two available indicators are mapped with `local_clusters` (Queen weights, 999 permutations, seed 42), the helper shared with the hot-spot analysis. Only locations with pseudo p < 0.05 receive a cluster label; all others are `ns` (not significant). HH and LL indicate similar local values, whereas HL and LH identify spatial outliers.

In [5]:
COLORS = {'HH': '#b2182b', 'LL': '#2166ac', 'HL': '#ef8a62', 'LH': '#67a9cf', 'ns': '#d9d9d9'}
LEGEND = {
    'HH': 'High-High (Hot spot)',
    'LL': 'Low-Low (Cold spot)',
    'HL': 'High-Low (Outlier)',
    'LH': 'Low-High (Outlier)',
    'ns': 'Not significant',
}
significant_agebs = []
for column in list(samples)[:2]:
    sample = samples[column].copy()
    weights = build_weights(sample, kind='queen')
    sample = sample.join(local_clusters(sample, column, weights, permutations=999, alpha=0.05, seed=SEED))
    significant_agebs.append(
        sample.loc[sample['cluster'].ne('ns'), [column, 'cluster', 'p_sim']].assign(indicator=column)
    )
    fig, ax = plt.subplots(figsize=(8, 8))
    for cluster, color in COLORS.items():
        sample.loc[sample['cluster'].eq(cluster)].plot(
            ax=ax, color=color, edgecolor='white', linewidth=.15, label=LEGEND.get(cluster, cluster)
        )
    ax.set_title(f"LISA clusters: {INDICATORS[column]} (p < 0.05)")
    ax.set_axis_off(); ax.legend(loc='lower left'); fig.tight_layout()
    fig.savefig(MAPS / f'32_lisa_{column}.png', dpi=180, bbox_inches='tight')
    plt.close(fig)
significant_agebs = pd.concat(significant_agebs).reset_index()
significant_agebs_sorted = significant_agebs.sort_values(['indicator', 'cluster', 'cvegeo'])
significant_agebs_sorted.to_csv(MAPS / '32_significant_agebs.csv', index=False)
display(significant_agebs_sorted)

,cvegeo,business_density_km2,cluster,p_sim,indicator,crime_rate_per_1k
92,0900600010700,781.544256,HH,0.021,business_density_km2,NaN
93,090060001072A,841.708543,HH,0.018,business_density_km2,NaN
97,0900700012140,1006.690029,HH,0.046,business_density_km2,NaN
99,0900700012719,1083.718541,HH,0.047,business_density_km2,NaN
100,0900700014217,1017.341040,HH,0.049,business_density_km2,NaN
...,...,...,...,...,...,...
1017,0901700011469,NaN,LL,0.016,crime_rate_per_1k,2.949853
1018,0901700011473,NaN,LL,0.001,crime_rate_per_1k,2.540220
1019,0901700011488,NaN,LL,0.002,crime_rate_per_1k,3.707136
1020,0901700011505,NaN,LL,0.007,crime_rate_per_1k,0.397931


## Interpretation guide

A small pseudo p-value indicates that the observed spatial pattern is unusual under random permutation, not that neighboring conditions cause one another. Differences between Queen and KNN-6 show sensitivity to the neighborhood definition. Results are also subject to temporal mismatch between Census 2020 and DENUE (May 2024), small population denominators, the modifiable areal unit problem, and the temporal coverage of the crime source (the FGJ 2024 extract covers investigation filings from 2024-01-01 to 2024-07-31; later months are unavailable rather than observed zero-crime periods).